# DesignBridge 3D Pipeline — ControlNet-Depth 版

**流程：** 設計圖 → Depth Anything V2 → **ControlNet-Depth 生成多視角** → InstantSplat 3D 重建

ControlNet 用深度圖引導 AI 補齊每個旋轉視角，比純幾何 warp 品質高很多。

**預計時間：** 安裝 10 分鐘、推理 10–15 分鐘 ｜ **需求：** T4 GPU

## Step 0 — 確認 GPU

In [ ]:
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    raise RuntimeError('⛔ 請到 執行階段 → 變更執行階段類型 → T4 GPU')

## Step 1 — 安裝 InstantSplat（約 5–8 分鐘）

In [ ]:
import os
if not os.path.exists('/content/InstantSplat'):
    !git clone --recursive https://github.com/NVlabs/InstantSplat.git /content/InstantSplat
else:
    print('already cloned')
%cd /content/InstantSplat

In [ ]:
!pip install -q -r requirements.txt
print('✅ requirements installed')

In [ ]:
import os
os.environ['CUDA_HOME'] = '/usr/local/cuda'
os.environ['MAX_JOBS'] = '4'
print('編譯 simple-knn ...')
!pip install -q submodules/simple-knn
print('編譯 diff-gaussian-rasterization ...')
!pip install -q submodules/diff-gaussian-rasterization
print('✅ CUDA extensions compiled')

In [ ]:
# Patch torch.load（PyTorch 2.4+ 安全限制）
import re
from pathlib import Path
patched = []
for py_file in Path('/content/InstantSplat').rglob('*.py'):
    try:
        text = py_file.read_text()
    except Exception:
        continue
    if 'torch.load' not in text:
        continue
    new_text = re.sub(
        r'torch\.load\(([^)]+)\)',
        lambda m: m.group(0) if 'weights_only' in m.group(1)
                  else f'torch.load({m.group(1)}, weights_only=False)',
        text,
    )
    if new_text != text:
        py_file.write_text(new_text)
        patched.append(py_file.name)
print(f'✅ Patched {len(patched)} files: {patched}')

## Step 2 — 下載 MASt3R checkpoint（約 1 GB）

In [ ]:
import os
ckpt_dir  = '/content/InstantSplat/mast3r/checkpoints'
ckpt_name = 'MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth'
ckpt_path = f'{ckpt_dir}/{ckpt_name}'
os.makedirs(ckpt_dir, exist_ok=True)
if not os.path.exists(ckpt_path) or os.path.getsize(ckpt_path) < 100_000_000:
    if os.path.exists(ckpt_path):
        os.remove(ckpt_path)
    print('下載中...')
    !wget -L -c --show-progress \
        -O "{ckpt_path}" \
        "https://download.europe.naverlabs.com/ComputerVision/MASt3R/{ckpt_name}"
size_gb = os.path.getsize(ckpt_path) / 1e9
print(f'大小：{size_gb:.2f} GB')
print('✅ OK' if size_gb > 0.5 else '❌ 下載失敗')

## Step 3 — 安裝 Depth Anything + ControlNet 依賴

> **⚠️ 跑完這個 cell 後請按「Runtime → Restart runtime」，然後從 Step 4 開始繼續**

In [ ]:
!pip install -q transformers accelerate diffusers xformers
print('✅ dependencies installed')

## Step 4 — 定義工具函式

In [ ]:
import numpy as np
from pathlib import Path
from PIL import Image
import shutil


def _fill_holes(arr, hole_mask):
    if not hole_mask.any():
        return arr
    try:
        from scipy.ndimage import distance_transform_edt
        _, idx = distance_transform_edt(hole_mask, return_indices=True)
        filled = arr.copy()
        filled[hole_mask] = arr[tuple(idx[:, hole_mask])]
        return filled
    except ImportError:
        return arr


def warp_depth_only(depth_arr, angle_deg, focal_scale=0.7):
    """把深度圖旋轉到新視角，回傳 float32 [0,1] array"""
    H, W = depth_arr.shape
    fx = fy = W * focal_scale
    cx, cy = W / 2.0, H / 2.0

    u_grid, v_grid = np.meshgrid(np.arange(W, dtype=np.float32), np.arange(H, dtype=np.float32))
    d_metric = depth_arr.astype(np.float32) * 5.0 + 0.1
    X = (u_grid - cx) * d_metric / fx
    Y = (v_grid - cy) * d_metric / fy
    Z = d_metric

    theta = np.radians(angle_deg)
    c, s = float(np.cos(theta)), float(np.sin(theta))
    X_r, Y_r, Z_r = c*X + s*Z, Y, -s*X + c*Z

    valid = Z_r > 0.1
    u_int = np.round(np.where(valid, fx * X_r / Z_r + cx, -1)).astype(np.int32)
    v_int = np.round(np.where(valid, fy * Y_r / Z_r + cy, -1)).astype(np.int32)
    in_bounds = valid & (u_int >= 0) & (u_int < W) & (v_int >= 0) & (v_int < H)

    d_reproj = np.clip((Z_r - 0.1) / 5.0, 0, 1)
    flat = np.where(in_bounds.ravel())[0]
    u_f, v_f = u_int.ravel()[flat], v_int.ravel()[flat]
    d_f = d_reproj.ravel()[flat]

    order = np.argsort(-d_f)
    new_depth = np.zeros((H, W), dtype=np.float32)
    new_depth[v_f[order], u_f[order]] = d_f[order]

    return _fill_holes(new_depth, new_depth == 0)


print('✅ warp functions ready')

## Step 5 — 上傳圖片
選方法 A（上傳）或方法 B（合成測試圖）

In [ ]:
# 方法 A：上傳自己的室內設計圖
from google.colab import files
uploaded = files.upload()
if uploaded:
    INPUT_IMAGE = list(uploaded.keys())[0]
    print(f'✅ 上傳：{INPUT_IMAGE}')
else:
    print('沒有上傳，請用方法 B')

In [ ]:
# 方法 B：合成測試圖（跳過方法 A 時用）
from PIL import Image, ImageDraw
W, H = 512, 512
img = Image.new('RGB', (W, H), '#c8b8a0')
draw = ImageDraw.Draw(img)
draw.rectangle([0, H*2//3, W, H], fill='#8B7355')
draw.rectangle([0, 0, W, H*2//3], fill='#D4C5B0')
draw.rectangle([W//6, H//2, W*5//6, H*2//3], fill='#4A5568')
draw.rectangle([W//6, H*5//12, W*5//6, H//2], fill='#718096')
draw.rectangle([W//3, H//8, W*2//3, H*5//12], fill='#BEE3F8')
draw.ellipse([W//2-30, 0, W//2+30, 40], fill='#FEF3C7')
INPUT_IMAGE = '/content/test_room.png'
img.save(INPUT_IMAGE)
print(f'✅ 合成測試圖：{INPUT_IMAGE}')
img

## Step 6 — Depth Anything V2 深度估測

In [ ]:
from transformers import pipeline as hf_pipeline
import numpy as np, matplotlib.pyplot as plt
from PIL import Image

print('載入 Depth Anything V2 Small...')
depth_estimator = hf_pipeline(
    task='depth-estimation',
    model='depth-anything/Depth-Anything-V2-Small-hf',
    device=0,
)
rgb_image = Image.open(INPUT_IMAGE).convert('RGB')
depth_raw = np.array(depth_estimator(rgb_image)['depth'], dtype=np.float32)
depth_norm = (depth_raw - depth_raw.min()) / (depth_raw.max() - depth_raw.min() + 1e-6)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(rgb_image); axes[0].set_title('輸入圖'); axes[0].axis('off')
axes[1].imshow(depth_norm, cmap='plasma'); axes[1].set_title('Depth Map'); axes[1].axis('off')
plt.tight_layout(); plt.show()
print(f'✅ depth shape={depth_norm.shape}')

## Step 6b — 載入 ControlNet-Depth 模型

首次下載約 **5 GB**（SD1.5 + ControlNet）

In [ ]:
import torch
from diffusers import StableDiffusionControlNetImg2ImgPipeline, ControlNetModel

print('載入 ControlNet-Depth...')
controlnet = ControlNetModel.from_pretrained(
    'lllyasviel/control_v11f1p_sd15_depth',
    torch_dtype=torch.float16,
)
print('載入 SD1.5 (img2img 模式)...')
cn_pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    controlnet=controlnet,
    torch_dtype=torch.float16,
    safety_checker=None,
).to('cuda')

try:
    cn_pipe.enable_xformers_memory_efficient_attention()
    print('xformers 省記憶體模式 ON')
except Exception:
    cn_pipe.enable_attention_slicing()
    print('attention slicing 省記憶體模式 ON')

print('✅ ControlNet-Depth img2img 就緒')
print(f'VRAM 使用：{torch.cuda.memory_allocated()/1e9:.1f} GB')

## Step 7 — ControlNet-Depth 生成多視角圖

把深度圖旋轉到 4 個角度，再讓 ControlNet-Depth 根據深度結構生成完整視角圖。

**`PROMPT`** 填入圖片的場景描述（越準確效果越好）

In [ ]:
import os, shutil
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# ── 設定 ────────────────────────────────────────────────
PROMPT   = 'interior room design, modern style, high quality, realistic lighting'
N_STEPS  = 25      # 推理步數（越高越好但越慢）
GUIDANCE = 7.5     # classifier-free guidance scale
CN_SCALE = 0.9     # ControlNet 深度圖影響強度（0~1）
ANGLES   = [-25.0, -12.0, 12.0, 25.0]  # 旋轉角度（度）
# ────────────────────────────────────────────────────────

SOURCE   = '/content/splat_views'
VIEW_DIR = f'{SOURCE}/images'
os.makedirs(VIEW_DIR, exist_ok=True)

# 原圖作為 view 0
orig_out = f'{VIEW_DIR}/0000.png'
shutil.copy(INPUT_IMAGE, orig_out)
image_paths = [orig_out]

W_orig, H_orig = Image.open(INPUT_IMAGE).size

for i, angle in enumerate(ANGLES, start=1):
    print(f'\n生成視角 {angle:+.0f}°...')

    # 1. 深度圖旋轉到新角度
    warped_depth = warp_depth_only(depth_norm, angle)

    # 2. 轉成 ControlNet 需要的 RGB 深度圖（三通道）
    depth_rgb = Image.fromarray(
        (warped_depth * 255).clip(0, 255).astype(np.uint8)
    ).convert('RGB').resize((W_orig, H_orig))

    # 3. ControlNet-Depth 生成
    result = cn_pipe(
        prompt=PROMPT,
        negative_prompt='blurry, low quality, distorted, ugly',
        image=depth_rgb,
        num_inference_steps=N_STEPS,
        guidance_scale=GUIDANCE,
        controlnet_conditioning_scale=CN_SCALE,
    ).images[0]

    out_path = f'{VIEW_DIR}/{i:04d}.png'
    result.save(out_path)
    image_paths.append(out_path)
    print(f'  ✅ 儲存：{out_path}')

print(f'\n✅ 共生成 {len(image_paths)} 張視角圖')

# 顯示結果
labels = ['原圖 (0°)'] + [f'{a:+.0f}°' for a in ANGLES]
fig, axes = plt.subplots(1, len(image_paths), figsize=(4*len(image_paths), 4))
for ax, path, label in zip(axes, image_paths, labels):
    ax.imshow(Image.open(path)); ax.set_title(label, fontsize=10); ax.axis('off')
plt.suptitle('ControlNet-Depth 多視角生成結果', fontsize=13)
plt.tight_layout(); plt.show()

## Step 8 — InstantSplat 3D 重建（約 5–10 分鐘）

用 ControlNet 生成的高品質視角圖做 3D 重建。

In [ ]:
import os, subprocess

# 釋放 ControlNet VRAM（給 InstantSplat 用）
import torch, gc
del cn_pipe, controlnet
gc.collect()
torch.cuda.empty_cache()
print(f'釋放後 VRAM：{torch.cuda.memory_allocated()/1e9:.1f} GB')

ISP_DIR = '/content/InstantSplat'
OUTPUT  = '/content/splat_output'
N_VIEWS = len(image_paths)
CKPT    = '/content/InstantSplat/mast3r/checkpoints/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth'
ITERS   = 2000  # 比之前 1000 多一倍，品質更好

os.makedirs(OUTPUT, exist_ok=True)

def run_step(cmd, label, timeout=900):
    print(f'\n▶ {label} ...')
    r = subprocess.run(cmd, cwd=ISP_DIR, capture_output=True, text=True, timeout=timeout)
    if r.returncode != 0:
        print(f'❌ 失敗：'); print(r.stderr[-1200:])
        return False
    print(f'✅ 完成')
    return True

common = ['-s', SOURCE, '-m', OUTPUT, '--n_views', str(N_VIEWS)]

ok = run_step(
    ['python', 'init_geo.py'] + common + [
        '--focal_avg', '--co_vis_dsp', '--conf_aware_ranking', '--ckpt_path', CKPT,
    ], 'init_geo.py（MASt3R 相機姿態估測）', timeout=300,
)
if ok:
    ok = run_step(
        ['python', 'train.py'] + common + [
            '-r', '1', '--iterations', str(ITERS), '--pp_optimizer', '--optim_pose',
        ], f'train.py（3DGS 訓練 {ITERS} 迭代）', timeout=900,
    )
if ok:
    ok = run_step(
        ['python', 'render.py'] + common + [
            '-r', '1', '--iterations', str(ITERS), '--skip_train',
        ], 'render.py（輸出渲染視角）', timeout=300,
)
if ok:
    print('\n🎉 InstantSplat 全流程完成！')

## Step 9 — 確認輸出

In [ ]:
import glob, os
from PIL import Image
import matplotlib.pyplot as plt

ply_files = glob.glob(f'{OUTPUT}/**/point_cloud.ply', recursive=True)
if ply_files:
    PLY_PATH = ply_files[0]
    print(f'✅ {PLY_PATH}')
    print(f'   大小：{os.path.getsize(PLY_PATH)/1e6:.1f} MB')
else:
    print('❌ 找不到 point_cloud.ply'); PLY_PATH = None

render_imgs = sorted(glob.glob(f'{OUTPUT}/**/*.png', recursive=True))[:6]
if render_imgs:
    fig, axes = plt.subplots(1, len(render_imgs), figsize=(4*len(render_imgs), 4))
    if len(render_imgs) == 1: axes = [axes]
    for ax, p in zip(axes, render_imgs):
        ax.imshow(Image.open(p)); ax.axis('off')
    plt.suptitle('渲染結果', fontsize=13)
    plt.tight_layout(); plt.show()

## Step 10 — 下載 .ply

In [ ]:
from google.colab import files
import shutil, os
if PLY_PATH and os.path.exists(PLY_PATH):
    dl = '/content/point_cloud.ply'
    shutil.copy(PLY_PATH, dl)
    print(f'下載 {os.path.getsize(dl)/1e6:.1f} MB...')
    files.download(dl)
    print('✅ 下載完成')
    print('\n放到本機：')
    print('  cp ~/Downloads/point_cloud.ply <專案路徑>/artifacts/splat/latest.ply')
else:
    print('❌ 請確認 Step 8 成功')